# Chapter 18: Agent Harness — The GAS Orchestration Engine

> **Part 5: Agentic AI Systems** | Guardian Angel System (GAS) — Multi-Agent Diabetic Patient Supervision

---

## 🎯 Chapter Goal

By the end of this chapter you will:

1. **Build** a production `GASHarness` class — the "OS" for the GAS agents
2. **Implement** 5 GAS tools with error handling and logging
3. **Add** safety guardrails: block unsafe insulin dose recommendations
4. **Instrument** observability: log every tool call with timestamp, input, output, latency
5. **Run** the harness end-to-end on a hypoglycemia scenario

### What is an Agent Harness?

An agent harness is the **runtime infrastructure** that wraps an LLM agent:

```
┌─────────────────────────────────────────────────────────┐
│                    GASHarness                           │
│                                                         │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌────────┐   │
│  │  Context │  │  Tool    │  │ Guard-   │  │ Observ-│   │
│  │  Manager │  │ Dispatch │  │  rails   │  │ ability│   │
│  └──────────┘  └──────────┘  └──────────┘  └────────┘   │
│                       │                                 │
│                  ┌────▼────┐                            │
│                  │   LLM   │                            │
│                  └─────────┘                            │
└─────────────────────────────────────────────────────────┘
```

Think of it as the **operating system** for your agent — it handles all the infrastructure so the LLM can focus on reasoning.

## 📖 Theory Recap

### What is an Agent Harness?

The harness is the layer between the LLM and the real world. It provides:

| Component | Responsibility |
|-----------|---------------|
| **Context Manager** | Keeps the LLM context within token limits |
| **Tool Dispatcher** | Routes tool calls to implementations, handles errors |
| **Guardrails** | Validates inputs and outputs for safety |
| **Observability** | Logs, traces, and metrics for debugging |
| **Human-in-the-Loop** | Escalation patterns for critical decisions |

### Context Window Management

LLMs have finite context windows. Three strategies:

1. **Sliding window**: Keep only the last N messages
2. **Summarization**: Compress old messages into a summary
3. **Hierarchical**: Keep important messages, compress the rest

For GAS, we use a hybrid: keep the last 10 messages + a running summary of older messages.

### Tool Dispatch and Error Handling

Tools fail. The harness must handle:
- **Timeout**: Tool takes too long → return error, continue
- **Exception**: Tool raises exception → log, return error message
- **Invalid args**: Tool called with wrong arguments → validate before calling
- **Retry**: Some failures are transient → retry with exponential backoff

### Guardrails

Guardrails are safety filters applied to LLM outputs:

```
LLM Output → [Input Guardrail] → [Output Guardrail] → Final Response
```

For GAS, the critical guardrail: **never recommend a specific insulin dose without Safety Critic approval**.

### Observability

Every tool call should be logged with:
- Timestamp
- Tool name and arguments
- Result (or error)
- Latency (ms)
- Agent that called it

This creates an **audit trail** — critical for medical AI.

## 🔨 Build It

### Step 1: Install Dependencies

In [ ]:
# Install required packages
# %pip install openai python-dotenv

### Step 2: Environment Setup

In [ ]:
import os
import json
import time
import re
import datetime
import traceback
from typing import TypedDict, List, Dict, Any, Optional, Tuple, Callable
from dataclasses import dataclass, field
from collections import deque

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL

### Step 3: Reuse Core Data Classes

In [ ]:
# ─── Core Data Classes (from Chapter 15) ─────────────────────────────────────

@dataclass
class PatientProfile:
    patient_id: str
    name: str
    age: int
    diabetes_type: str
    target_glucose_low: float
    target_glucose_high: float
    insulin_sensitivity: float
    carb_ratio: float
    emergency_contact: str = "caregiver@example.com"


@dataclass
class GlucoseReading:
    value: float
    timestamp: datetime.datetime
    trend: str
    sensor_id: str
    confidence: float = 1.0


class GASState(TypedDict):
    patient: Any
    current_reading: Any
    memory: Any
    alert_level: str
    recommendation: str
    safety_approved: bool
    caregiver_notified: bool
    agent_messages: List[dict]
    next_agent: str


# Alice — our synthetic T1D patient
alice = PatientProfile(
    patient_id="patient-001",
    name="Alice",
    age=34,
    diabetes_type="T1D",
    target_glucose_low=70.0,
    target_glucose_high=180.0,
    insulin_sensitivity=50.0,
    carb_ratio=15.0,
    emergency_contact="bob.smith@example.com"
)

print(f"✅ Core data classes defined. Patient: {alice.name}")

### Step 4: Define the 5 GAS Tools

In [ ]:
# ─── GAS Tool Implementations ─────────────────────────────────────────────────

# Simulated data stores
_CGM_DATA = {"patient-001": {"value": 45.0, "trend": "falling", "sensor_id": "dexcom-g7-001"}}
_PATIENT_HISTORY = {
    "patient-001": [
        {"date": "2024-01-10", "event": "hypoglycemia", "value": 48, "time": "03:00"},
        {"date": "2024-01-11", "event": "hyperglycemia", "value": 285, "time": "21:00"},
        {"date": "2024-01-12", "event": "hypoglycemia", "value": 55, "time": "15:30"},
    ]
}
_ALERT_LOG = []


def tool_read_glucose(patient_id: str) -> dict:
    """Tool 1: Read current CGM glucose value."""
    data = _CGM_DATA.get(patient_id, {"value": 100.0, "trend": "stable"})
    return {
        "patient_id": patient_id,
        "glucose_value": data["value"],
        "trend": data["trend"],
        "unit": "mg/dL",
        "timestamp": datetime.datetime.now().isoformat(),
        "sensor_id": data.get("sensor_id", "unknown")
    }


def tool_get_patient_history(patient_id: str, n_events: int = 5) -> dict:
    """Tool 2: Get recent patient history."""
    history = _PATIENT_HISTORY.get(patient_id, [])
    return {
        "patient_id": patient_id,
        "recent_events": history[-n_events:],
        "total_events": len(history)
    }


def tool_lookup_guideline(condition: str) -> dict:
    """Tool 3: Look up clinical guideline for a condition."""
    guidelines = {
        "hypoglycemia": {
            "condition": "Hypoglycemia (< 70 mg/dL)",
            "action": "Apply Rule of 15: 15g fast-acting carbs, wait 15 min, recheck.",
            "source": "ADA 2024"
        },
        "severe_hypoglycemia": {
            "condition": "Severe Hypoglycemia (< 54 mg/dL)",
            "action": "If conscious: 15-20g fast carbs. If unconscious: glucagon + call 911.",
            "source": "ADA 2024"
        },
        "hyperglycemia": {
            "condition": "Hyperglycemia (> 180 mg/dL)",
            "action": "Check ketones. Calculate correction dose = (BG - target) / ISF.",
            "source": "ADA 2024"
        },
        "dka": {
            "condition": "DKA Risk (BG > 250 + ketones)",
            "action": "EMERGENCY: Call 911. Do not attempt home treatment.",
            "source": "ADA 2024"
        }
    }
    key = condition.lower().replace(" ", "_")
    return guidelines.get(key, {
        "condition": condition,
        "action": "Consult healthcare provider.",
        "source": "ADA 2024"
    })


def tool_calculate_insulin_dose(
    current_glucose: float,
    target_glucose: float,
    insulin_sensitivity: float,
    insulin_on_board: float = 0.0
) -> dict:
    """Tool 4: Calculate correction insulin dose."""
    if current_glucose < 70:
        return {
            "error": "Cannot calculate insulin dose for hypoglycemia. Treat low first.",
            "safe": False
        }

    raw_dose = (current_glucose - target_glucose) / insulin_sensitivity
    adjusted_dose = max(0, raw_dose - insulin_on_board)
    rounded_dose = round(adjusted_dose * 2) / 2  # Round to nearest 0.5 unit

    return {
        "current_glucose": current_glucose,
        "target_glucose": target_glucose,
        "insulin_sensitivity": insulin_sensitivity,
        "raw_dose": round(raw_dose, 2),
        "insulin_on_board": insulin_on_board,
        "adjusted_dose": round(adjusted_dose, 2),
        "recommended_dose": rounded_dose,
        "unit": "units",
        "safe": True,
        "warning": "REQUIRES SAFETY CRITIC APPROVAL before administration"
    }


def tool_send_alert(level: str, message: str, patient_id: str) -> dict:
    """Tool 5: Send alert to caregiver."""
    alert = {
        "status": "sent",
        "alert_id": f"alert-{len(_ALERT_LOG)+1:04d}",
        "level": level,
        "message": message,
        "patient_id": patient_id,
        "recipient": alice.emergency_contact,
        "timestamp": datetime.datetime.now().isoformat(),
        "delivery_method": "SMS + push notification"
    }
    _ALERT_LOG.append(alert)
    return alert


# Tool registry
TOOL_REGISTRY: Dict[str, Callable] = {
    "read_glucose": tool_read_glucose,
    "get_patient_history": tool_get_patient_history,
    "lookup_guideline": tool_lookup_guideline,
    "calculate_insulin_dose": tool_calculate_insulin_dose,
    "send_alert": tool_send_alert,
}

# OpenAI tool definitions
GAS_TOOL_SCHEMAS = [
    {
        "type": "function",
        "function": {
            "name": "read_glucose",
            "description": "Read the current CGM glucose value and trend for a patient.",
            "parameters": {
                "type": "object",
                "properties": {"patient_id": {"type": "string"}},
                "required": ["patient_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_patient_history",
            "description": "Get recent clinical history for a patient.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {"type": "string"},
                    "n_events": {"type": "integer", "default": 5}
                },
                "required": ["patient_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_guideline",
            "description": "Look up ADA clinical guideline for a condition (hypoglycemia, hyperglycemia, dka, severe_hypoglycemia).",
            "parameters": {
                "type": "object",
                "properties": {"condition": {"type": "string"}},
                "required": ["condition"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_insulin_dose",
            "description": "Calculate correction insulin dose. REQUIRES Safety Critic approval before use.",
            "parameters": {
                "type": "object",
                "properties": {
                    "current_glucose": {"type": "number"},
                    "target_glucose": {"type": "number"},
                    "insulin_sensitivity": {"type": "number"},
                    "insulin_on_board": {"type": "number", "default": 0.0}
                },
                "required": ["current_glucose", "target_glucose", "insulin_sensitivity"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_alert",
            "description": "Send an alert to the patient's caregiver.",
            "parameters": {
                "type": "object",
                "properties": {
                    "level": {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                    "message": {"type": "string"},
                    "patient_id": {"type": "string"}
                },
                "required": ["level", "message", "patient_id"]
            }
        }
    }
]

print(f"✅ Defined {len(TOOL_REGISTRY)} GAS tools:")
for name in TOOL_REGISTRY:
    print(f"  🔧 {name}")

### Step 5: Build the Observability Layer

In [ ]:
# ─── Observability: Tool Call Logger ─────────────────────────────────────────

@dataclass
class ToolCallLog:
    """A single tool call log entry."""
    timestamp: str
    agent: str
    tool_name: str
    args: dict
    result: Any
    latency_ms: float
    success: bool
    error: Optional[str] = None


class ObservabilityLogger:
    """Logs all tool calls for audit and debugging."""

    def __init__(self):
        self.logs: List[ToolCallLog] = []
        self._session_start = datetime.datetime.now()

    def log(self, log_entry: ToolCallLog):
        self.logs.append(log_entry)
        status = "✅" if log_entry.success else "❌"
        print(f"  {status} [{log_entry.timestamp}] {log_entry.agent}.{log_entry.tool_name} "
              f"({log_entry.latency_ms:.1f}ms)")
        if log_entry.error:
            print(f"     ERROR: {log_entry.error}")

    def get_summary(self) -> dict:
        """Get summary statistics for the session."""
        if not self.logs:
            return {}
        total = len(self.logs)
        successful = sum(1 for l in self.logs if l.success)
        avg_latency = sum(l.latency_ms for l in self.logs) / total
        tool_counts = {}
        for l in self.logs:
            tool_counts[l.tool_name] = tool_counts.get(l.tool_name, 0) + 1
        return {
            "total_calls": total,
            "successful": successful,
            "failed": total - successful,
            "success_rate": f"{successful/total*100:.0f}%",
            "avg_latency_ms": round(avg_latency, 1),
            "tool_call_counts": tool_counts,
            "session_duration_s": round((datetime.datetime.now() - self._session_start).total_seconds(), 1)
        }

    def print_audit_trail(self):
        """Print the full audit trail."""
        print("\n📋 AUDIT TRAIL")
        print("=" * 70)
        for i, log in enumerate(self.logs, 1):
            status = "✅" if log.success else "❌"
            print(f"{i:2d}. {status} {log.timestamp} | {log.agent:15s} | {log.tool_name:25s} | {log.latency_ms:6.1f}ms")
            if log.error:
                print(f"    ERROR: {log.error}")


print("✅ ObservabilityLogger defined")

### Step 6: Build the Safety Guardrails

In [ ]:
# ─── Safety Guardrails ────────────────────────────────────────────────────────

class SafetyGuardrails:
    """
    Safety guardrails for the GAS system.
    Validates LLM outputs before they reach the patient.
    """

    # Patterns that indicate unsafe insulin dose recommendations
    UNSAFE_PATTERNS = [
        r"take\s+\d+(\.\d+)?\s+unit",          # "take 3 units"
        r"inject\s+\d+(\.\d+)?\s+unit",         # "inject 2.5 units"
        r"administer\s+\d+(\.\d+)?\s+unit",     # "administer 4 units"
        r"dose\s+of\s+\d+(\.\d+)?\s+unit",      # "dose of 3 units"
        r"give\s+\d+(\.\d+)?\s+unit",           # "give 2 units"
        r"bolus\s+\d+(\.\d+)?\s+unit",          # "bolus 5 units"
    ]

    # Patterns that indicate emergency situations requiring immediate escalation
    EMERGENCY_PATTERNS = [
        r"call\s+911",
        r"emergency\s+services",
        r"unconscious",
        r"seizure",
        r"dka",
        r"diabetic\s+ketoacidosis",
    ]

    def __init__(self, safety_approved: bool = False):
        self.safety_approved = safety_approved

    def check_input(self, user_message: str) -> Tuple[bool, str]:
        """
        Validate input before sending to LLM.
        Returns (is_safe, reason).
        """
        # Check for prompt injection attempts
        injection_patterns = [
            r"ignore\s+previous\s+instructions",
            r"forget\s+your\s+instructions",
            r"you\s+are\s+now\s+a",
            r"jailbreak",
        ]
        for pattern in injection_patterns:
            if re.search(pattern, user_message.lower()):
                return False, f"Potential prompt injection detected: {pattern}"
        return True, "Input is safe"

    def check_output(self, response: str) -> Tuple[str, bool, List[str]]:
        """
        Validate and filter LLM output.
        Returns (filtered_response, is_safe, warnings).
        """
        warnings = []
        filtered = response
        is_safe = True

        # Check for unsafe insulin dose recommendations
        if not self.safety_approved:
            for pattern in self.UNSAFE_PATTERNS:
                if re.search(pattern, response.lower()):
                    warnings.append(f"Unsafe insulin dose recommendation detected (pattern: {pattern})")
                    # Replace with safe version
                    filtered = re.sub(
                        pattern,
                        "[DOSE REDACTED — requires Safety Critic approval]",
                        filtered,
                        flags=re.IGNORECASE
                    )
                    is_safe = False

        # Check for emergency patterns — these should trigger escalation
        for pattern in self.EMERGENCY_PATTERNS:
            if re.search(pattern, response.lower()):
                warnings.append(f"Emergency situation detected: {pattern}")

        return filtered, is_safe, warnings

    def check_tool_call(self, tool_name: str, args: dict) -> Tuple[bool, str]:
        """
        Validate a tool call before execution.
        Returns (is_allowed, reason).
        """
        # Block insulin dose calculation without safety approval
        if tool_name == "calculate_insulin_dose" and not self.safety_approved:
            # Allow calculation but flag it
            return True, "WARNING: Insulin dose calculation requires Safety Critic approval before use"

        # Block alerts with invalid levels
        if tool_name == "send_alert":
            valid_levels = ["low", "medium", "high", "critical"]
            if args.get("level") not in valid_levels:
                return False, f"Invalid alert level: {args.get('level')}. Must be one of {valid_levels}"

        return True, "Tool call is allowed"


# Test guardrails
guardrails = SafetyGuardrails(safety_approved=False)

print("🛡️  Safety Guardrails Test")
print("=" * 50)

# Test 1: Unsafe insulin recommendation
unsafe_response = "Based on the calculation, you should take 3.5 units of insulin now."
filtered, is_safe, warnings = guardrails.check_output(unsafe_response)
print(f"\nTest 1 — Unsafe insulin dose:")
print(f"  Original: {unsafe_response}")
print(f"  Filtered: {filtered}")
print(f"  Safe: {is_safe}, Warnings: {warnings}")

# Test 2: Safe recommendation
safe_response = "Apply Rule of 15: consume 15g fast-acting carbohydrates immediately."
filtered2, is_safe2, warnings2 = guardrails.check_output(safe_response)
print(f"\nTest 2 — Safe recommendation:")
print(f"  Response: {safe_response}")
print(f"  Safe: {is_safe2}, Warnings: {warnings2}")

# Test 3: Emergency detection
emergency_response = "Patient is unconscious. Call 911 immediately."
filtered3, is_safe3, warnings3 = guardrails.check_output(emergency_response)
print(f"\nTest 3 — Emergency detection:")
print(f"  Response: {emergency_response}")
print(f"  Safe: {is_safe3}, Warnings: {warnings3}")

### Step 7: Build the GASHarness

In [ ]:
# ─── GASHarness: The Agent Runtime ───────────────────────────────────────────

class GASHarness:
    """
    The GAS Agent Harness — the production runtime for all 5 GAS agents.

    Responsibilities:
    - Tool dispatch with error handling and retry
    - Safety guardrails (input + output filtering)
    - Context window management (sliding window + summarization)
    - Observability (audit logging)
    - Human-in-the-loop escalation
    """

    def __init__(
        self,
        tools: Dict[str, Callable],
        tool_schemas: List[dict],
        guardrails: SafetyGuardrails,
        logger: ObservabilityLogger,
        max_context_messages: int = 20,
        agent_name: str = "GASHarness"
    ):
        self.tools = tools
        self.tool_schemas = tool_schemas
        self.guardrails = guardrails
        self.logger = logger
        self.max_context_messages = max_context_messages
        self.agent_name = agent_name
        self._context_summary = ""

    def _dispatch_tool(self, tool_name: str, args: dict) -> Tuple[str, bool]:
        """
        Execute a tool with error handling and logging.
        Returns (result_json, success).
        """
        # Guardrail check
        allowed, reason = self.guardrails.check_tool_call(tool_name, args)
        if not allowed:
            error_result = json.dumps({"error": reason, "blocked_by": "guardrail"})
            self.logger.log(ToolCallLog(
                timestamp=datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3],
                agent=self.agent_name,
                tool_name=tool_name,
                args=args,
                result=error_result,
                latency_ms=0,
                success=False,
                error=reason
            ))
            return error_result, False

        # Execute tool
        start_time = time.time()
        try:
            if tool_name not in self.tools:
                raise ValueError(f"Unknown tool: {tool_name}")
            result = self.tools[tool_name](**args)
            result_json = json.dumps(result, indent=2, default=str)
            latency_ms = (time.time() - start_time) * 1000

            self.logger.log(ToolCallLog(
                timestamp=datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3],
                agent=self.agent_name,
                tool_name=tool_name,
                args=args,
                result=result,
                latency_ms=latency_ms,
                success=True
            ))
            return result_json, True

        except Exception as e:
            latency_ms = (time.time() - start_time) * 1000
            error_msg = f"{type(e).__name__}: {str(e)}"
            error_result = json.dumps({"error": error_msg})

            self.logger.log(ToolCallLog(
                timestamp=datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3],
                agent=self.agent_name,
                tool_name=tool_name,
                args=args,
                result=error_result,
                latency_ms=latency_ms,
                success=False,
                error=error_msg
            ))
            return error_result, False

    def _apply_guardrails(self, response: str) -> Tuple[str, bool, List[str]]:
        """Apply output guardrails. Returns (filtered_response, is_safe, warnings)."""
        return self.guardrails.check_output(response)

    def _manage_context(self, messages: List[dict]) -> List[dict]:
        """
        Manage context window size.
        Keep last N messages; summarize older ones.
        """
        if len(messages) <= self.max_context_messages:
            return messages

        # Keep system message + last N messages
        system_msgs = [m for m in messages if m.get("role") == "system"]
        non_system = [m for m in messages if m.get("role") != "system"]

        # Summarize old messages
        old_msgs = non_system[:-self.max_context_messages]
        recent_msgs = non_system[-self.max_context_messages:]

        if old_msgs:
            summary_text = f"[CONTEXT SUMMARY: {len(old_msgs)} earlier messages summarized. "
            summary_text += "Key events: " + "; ".join(
                m.get("content", "")[:50] for m in old_msgs[-3:]
            ) + "]"
            summary_msg = {"role": "system", "content": summary_text}
            return system_msgs + [summary_msg] + recent_msgs

        return system_msgs + recent_msgs

    def run_mock(
        self,
        state: GASState,
        scenario: str,
        max_steps: int = 5
    ) -> GASState:
        """
        Run the harness in mock mode (no LLM required).
        Simulates the full agent loop with tool calls.
        """
        patient_id = state["patient"].patient_id
        print(f"\n🤖 GASHarness — Mock Run")
        print(f"   Scenario: {scenario}")
        print(f"   Patient: {state['patient'].name}")
        print(f"   Max steps: {max_steps}")
        print("   Tool calls:")

        # Step 1: Read glucose
        result1, _ = self._dispatch_tool("read_glucose", {"patient_id": patient_id})
        glucose_data = json.loads(result1)
        glucose = glucose_data.get("glucose_value", 100)
        trend = glucose_data.get("trend", "stable")

        # Step 2: Get patient history
        result2, _ = self._dispatch_tool("get_patient_history", {"patient_id": patient_id})

        # Step 3: Look up guideline
        condition = "severe_hypoglycemia" if glucose < 54 else "hypoglycemia" if glucose < 70 else "hyperglycemia"
        result3, _ = self._dispatch_tool("lookup_guideline", {"condition": condition})
        guideline = json.loads(result3)

        # Step 4: Send alert if critical
        alert_level = "critical" if glucose < 54 else "high" if glucose < 70 else "medium"
        alert_msg = (
            f"URGENT: Patient {patient_id} glucose is {glucose} mg/dL ({trend}). "
            f"Action: {guideline.get('action', 'See guidelines')[:100]}"
        )
        result4, _ = self._dispatch_tool("send_alert", {
            "level": alert_level,
            "message": alert_msg,
            "patient_id": patient_id
        })

        # Build recommendation
        raw_recommendation = (
            f"ALERT: Glucose {glucose} mg/dL ({trend}). "
            f"{guideline.get('action', 'Consult provider.')} "
            f"Caregiver notified via {alert_level} alert."
        )

        # Apply guardrails
        filtered_rec, is_safe, warnings = self._apply_guardrails(raw_recommendation)

        if warnings:
            print(f"\n   ⚠️  Guardrail warnings: {warnings}")

        # Update state
        updated_state = dict(state)
        updated_state["alert_level"] = alert_level
        updated_state["recommendation"] = filtered_rec
        updated_state["safety_approved"] = is_safe
        updated_state["caregiver_notified"] = True
        updated_state["agent_messages"] = state["agent_messages"] + [
            {"agent": self.agent_name, "action": "run", "result": filtered_rec}
        ]
        updated_state["next_agent"] = "safety_critic" if not is_safe else "done"

        return updated_state

    def run_live(
        self,
        state: GASState,
        scenario: str,
        max_steps: int = 5
    ) -> GASState:
        """Run the harness with live OpenAI API."""
        system_prompt = (
            f"You are the GAS Monitor Agent for patient {state['patient'].name} "
            f"({state['patient'].age}yo {state['patient'].diabetes_type}). "
            f"Target range: {state['patient'].target_glucose_low}-{state['patient'].target_glucose_high} mg/dL. "
            "Use the available tools to assess the patient's glucose status and take appropriate action. "
            "Always prioritize patient safety."
        )

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Scenario: {scenario}. Assess and respond."}
        ]

        print(f"\n🤖 GASHarness — Live Run")
        print(f"   Scenario: {scenario}")
        print("   Tool calls:")

        final_response = ""
        for step in range(max_steps):
            messages = self._manage_context(messages)

            response = llm.chat_tools(messages, tools=self.tool_schemas, tool_choice="auto")

            msg = response.choices[0].message

            if not msg.tool_calls:
                final_response = msg.content or ""
                break

            messages.append(msg.model_dump() if hasattr(msg, "model_dump") else msg)
            tool_results = []

            for tc in msg.tool_calls:
                fn_name = tc.function.name
                fn_args = json.loads(tc.function.arguments)
                result, success = self._dispatch_tool(fn_name, fn_args)
                tool_results.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })

            messages.extend(tool_results)

        # Apply guardrails
        filtered_rec, is_safe, warnings = self._apply_guardrails(final_response)

        updated_state = dict(state)
        updated_state["recommendation"] = filtered_rec
        updated_state["safety_approved"] = is_safe
        updated_state["caregiver_notified"] = any(
            l.tool_name == "send_alert" and l.success for l in self.logger.logs
        )
        return updated_state

    def run(self, state: GASState, scenario: str, max_steps: int = 5) -> GASState:
        """Run the harness (mock or live based on TEST_MODE)."""
        if TEST_MODE:
            return self.run_mock(state, scenario, max_steps)
        else:
            return self.run_live(state, scenario, max_steps)


print("✅ GASHarness class defined")

### Step 8: Run the Harness — Hypoglycemia Scenario

In [ ]:
# ─── Initialize and Run the GASHarness ───────────────────────────────────────

# Initialize components
logger = ObservabilityLogger()
guardrails = SafetyGuardrails(safety_approved=False)

harness = GASHarness(
    tools=TOOL_REGISTRY,
    tool_schemas=GAS_TOOL_SCHEMAS,
    guardrails=guardrails,
    logger=logger,
    max_context_messages=20,
    agent_name="monitor_agent"
)

# Create initial GASState
initial_state: GASState = {
    "patient": alice,
    "current_reading": GlucoseReading(
        value=45.0,
        timestamp=datetime.datetime.now(),
        trend="falling",
        sensor_id="dexcom-g7-001"
    ),
    "memory": None,
    "alert_level": "none",
    "recommendation": "",
    "safety_approved": False,
    "caregiver_notified": False,
    "agent_messages": [],
    "next_agent": "monitor"
}

# Run the harness
SCENARIO = "Patient Alice's glucose just dropped to 45 mg/dL with a falling trend. This is a medical emergency."

final_state = harness.run(initial_state, SCENARIO, max_steps=5)

print(f"\n{'='*60}")
print("📊 GASHarness Run Results")
print(f"{'='*60}")
print(f"  Alert level: {final_state['alert_level']}")
print(f"  Safety approved: {final_state['safety_approved']}")
print(f"  Caregiver notified: {final_state['caregiver_notified']}")
print(f"  Next agent: {final_state['next_agent']}")
print(f"\n  Recommendation:")
print(f"  {final_state['recommendation']}")

In [ ]:
# ─── Print Observability Summary ──────────────────────────────────────────────

logger.print_audit_trail()

print("\n📈 Session Summary:")
summary = logger.get_summary()
for key, value in summary.items():
    print(f"  {key}: {value}")

### Step 9: Test Guardrail — Unsafe Insulin Dose

In [ ]:
# ─── Guardrail Test: Unsafe Insulin Dose Scenario ────────────────────────────

print("🛡️  Guardrail Test — Unsafe Insulin Dose Scenario")
print("=" * 60)

# Simulate a hyperglycemia scenario where LLM might recommend insulin
_CGM_DATA["patient-001"] = {"value": 280.0, "trend": "rising", "sensor_id": "dexcom-g7-001"}

hyper_state: GASState = {
    "patient": alice,
    "current_reading": GlucoseReading(
        value=280.0,
        timestamp=datetime.datetime.now(),
        trend="rising",
        sensor_id="dexcom-g7-001"
    ),
    "memory": None,
    "alert_level": "none",
    "recommendation": "",
    "safety_approved": False,
    "caregiver_notified": False,
    "agent_messages": [],
    "next_agent": "monitor"
}

# Test the calculate_insulin_dose tool directly
print("\nTest: calculate_insulin_dose tool")
dose_result, success = harness._dispatch_tool("calculate_insulin_dose", {
    "current_glucose": 280.0,
    "target_glucose": 120.0,
    "insulin_sensitivity": 50.0,
    "insulin_on_board": 0.0
})
dose_data = json.loads(dose_result)
print(f"  Calculated dose: {dose_data.get('recommended_dose')} units")
print(f"  Warning: {dose_data.get('warning')}")

# Test guardrail on a response that includes a specific dose
print("\nTest: Guardrail on response with specific dose")
unsafe_response = f"Based on the ISF calculation, you should inject {dose_data.get('recommended_dose')} units of insulin now."
filtered, is_safe, warnings = guardrails.check_output(unsafe_response)
print(f"  Original: {unsafe_response}")
print(f"  Filtered: {filtered}")
print(f"  Safe: {is_safe}")
print(f"  Warnings: {warnings}")

# Reset CGM data
_CGM_DATA["patient-001"] = {"value": 45.0, "trend": "falling", "sensor_id": "dexcom-g7-001"}

## 🏥 GAS Connection

The `GASHarness` is the **runtime that coordinates all 5 GAS agents**.

In the full GAS system (Chapter 20), the Supervisor Agent uses the harness to run each agent:

```python
# Full GAS pipeline (preview — Chapter 20)
harness = GASHarness(tools, schemas, guardrails, logger)

# Each agent runs through the harness
state = harness.run(state, agent="monitor")    # Read CGM, detect anomaly
state = harness.run(state, agent="advisor")    # RAG + memory recommendation
state = harness.run(state, agent="safety")     # Validate recommendation
state = harness.run(state, agent="notifier")   # Escalate if needed
```

### The Audit Trail is Critical for Medical AI

Every tool call is logged with:
- **Who** called it (which agent)
- **What** was called (tool name + args)
- **What** was returned (result)
- **When** (timestamp)
- **How long** it took (latency)

This audit trail is required for:
- **Regulatory compliance** (FDA, HIPAA)
- **Incident investigation** (what happened when Alice's glucose crashed?)
- **Model improvement** (which tool calls led to good/bad outcomes?)

## ✅ Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| 5 GAS Tools | read_glucose, get_patient_history, lookup_guideline, calculate_insulin_dose, send_alert |
| `ObservabilityLogger` | Logs every tool call with timestamp, latency, success/failure |
| `SafetyGuardrails` | Input validation, output filtering, unsafe dose detection |
| `GASHarness` | Full agent runtime: tool dispatch, guardrails, context management |
| Audit trail | Complete log of all tool calls for regulatory compliance |

### Key Concepts

- **Agent harness**: The "OS" for agents — handles infrastructure so LLM focuses on reasoning
- **Guardrails**: Safety filters that prevent dangerous outputs (e.g., unsanctioned insulin doses)
- **Observability**: Every action logged for audit, debugging, and improvement
- **Context management**: Sliding window + summarization to stay within token limits

### What's Next

**Chapter 19: Loop Engineering** — We'll build the production alert loop:
- 5 structural primitives: Generator, Verifier, Terminator, StateManager, Escalator
- Stagnation detection: if 3 iterations produce the same recommendation, escalate
- Adaptive budget: normal → 1 iteration; critical → 5 iterations + human
- The GAS Alert Loop that runs every 5 minutes

---
*Guardian Angel System | Part 5: Agentic AI | Chapter 18*